In [31]:
import gradio as gr
import json
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings


embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")

vectordb = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

with open('data/file_mapping.json', 'r') as fm:
    files = json.load(fm)
file_mapping = {f["id"]: f["path"] for f in files}

def on_semantic_search_click(query, k=3):
    if not query:
        return "Please enter a query."

    results = vectordb.similarity_search_with_score(query, k=k)

    output = ""
    for i, (doc, score) in enumerate(results, 1):
        content = doc.page_content
        metadata = doc.metadata

        output += f"{i}. Score={score:.3f}\n"
        output += f"Text: {content}\n"
        output += f"Translation: {metadata['tr_text']}\n"
        output += f"{metadata['lang']} to {metadata['tr_lang']}\n"
        output += f"Source: {file_mapping[metadata['file_id']]}\n\n"

    return output.strip()


with gr.Blocks() as app:
    gr.Markdown("## Semantic Search")
    with gr.Row():
        input_box = gr.Textbox(label="Enter your query", 
                               value="Когда же восходящее солнце пробило туман и осветило мертвеца",
                               lines=7)
        output_box = gr.Textbox(label="Result", lines=7)
    with gr.Row():
        k_input = gr.Number(label="Number of results (k)", value=3, precision=0)
    search_button = gr.Button("search", scale=0)

    search_button.click(
        fn=on_semantic_search_click,
        inputs=[input_box, k_input],
        outputs=output_box
    )

app.launch()

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.
